# Tema 11 - Filtros Activos

**Electronica General - 2o GIERM**

---

**Contenido:**
1. Introduccion y motivacion
2. Filtros de primer orden (paso bajo, paso alto)
3. Filtros de segundo orden: funcion de transferencia general
4. Aproximacion Butterworth (maximally flat)
5. Aproximacion Chebyshev (equiripple)
6. Aproximacion Bessel (fase lineal)
7. Topologia Sallen-Key
8. Sensibilidad
9. Comparacion de respuestas (Butterworth vs Chebyshev vs Bessel)
10. Respuesta al escalon
11. Seleccion del orden del filtro
12. Ubicacion de polos en el plano s
13. Ejercicios resueltos
14. Catalogo de tipos de ejercicio
15. Resumen y tabla de formulas clave

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from scipy import signal
from scipy.signal import butter, cheby1, bessel, freqs, tf2zpk
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6),
                      'figure.dpi': 100, 'axes.titlesize': 14})

# Paleta de colores
COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA     = '#cb181d'   # rojo - limites, chebyshev
COLOR_PUNTO     = '#238b45'   # verde - puntos, bessel
COLOR_BUTTER    = '#2171b5'   # azul - Butterworth
COLOR_CHEBY     = '#cb181d'   # rojo - Chebyshev
COLOR_BESSEL    = '#238b45'   # verde - Bessel
COLOR_FONDO     = '#f0f0f0'   # fondo claro
COLOR_ACCENT    = '#6a51a3'   # purpura - acentos

print('Librerias cargadas correctamente.')

---
## 1. Introduccion y motivacion

Un **filtro activo** es un circuito que utiliza **amplificadores operacionales** (op-amps) junto con resistencias y condensadores para seleccionar frecuencias de una senal.

**Ventajas frente a filtros pasivos (RC, RL, RLC):**
- No requieren inductores (pesados, costosos y no ideales a bajas frecuencias)
- Proporcionan **ganancia** (|H| > 1)
- Alta impedancia de entrada, baja de salida
- Facil cascadeo sin efecto de carga

**Tipos segun banda de paso:**

| Tipo | Deja pasar | Atenua |
|------|-----------|--------|
| Paso bajo (LP) | $\omega < \omega_c$ | $\omega > \omega_c$ |
| Paso alto (HP) | $\omega > \omega_c$ | $\omega < \omega_c$ |
| Paso banda (BP) | $\omega_1 < \omega < \omega_2$ | Fuera de banda |
| Rechazo de banda (BS) | Fuera de banda | $\omega_1 < \omega < \omega_2$ |

In [ ]:
# --- Respuesta ideal de los 4 tipos de filtro ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

titulos = ['Paso Bajo (LP)', 'Paso Alto (HP)', 'Paso Banda (BP)', 'Rechazo de Banda (BS)']
colores = [COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO, COLOR_ACCENT]

f = np.linspace(0, 10, 1000)
fc = 3.0
f1, f2 = 2.0, 5.0

# Respuestas ideales
resp_lp = np.where(f <= fc, 1.0, 0.0)
resp_hp = np.where(f >= fc, 1.0, 0.0)
resp_bp = np.where((f >= f1) & (f <= f2), 1.0, 0.0)
resp_bs = np.where((f >= f1) & (f <= f2), 0.0, 1.0)
respuestas = [resp_lp, resp_hp, resp_bp, resp_bs]

for ax, titulo, color, resp in zip(axes.flat, titulos, colores, respuestas):
    ax.fill_between(f, resp, alpha=0.25, color=color)
    ax.plot(f, resp, color=color, lw=2.5)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Frecuencia (kHz)')
    ax.set_ylabel('|H(jw)|')
    ax.set_ylim(-0.1, 1.3)
    ax.set_xlim(0, 10)

fig.suptitle('Respuestas ideales de filtros', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Filtros de primer orden

### 2.1 Paso bajo de primer orden

Funcion de transferencia:

$$H(s) = \frac{H_0}{1 + s/\omega_c} = \frac{H_0}{1 + j\omega/\omega_c}$$

donde $H_0$ es la ganancia DC y $\omega_c = 1/(RC)$ es la frecuencia de corte (-3 dB).

**Propiedades:**
- A $\omega = \omega_c$: $|H| = H_0/\sqrt{2}$ (caida de -3 dB)
- Pendiente de caida: **-20 dB/decada** (-6 dB/octava)
- Fase: de $0^\circ$ a $-90^\circ$, pasando por $-45^\circ$ en $\omega_c$

### 2.2 Paso alto de primer orden

$$H(s) = \frac{H_0 \cdot s/\omega_c}{1 + s/\omega_c}$$

Pendiente de subida: **+20 dB/decada** en la banda de transicion.

In [ ]:
# --- Diagrama de Bode: filtro paso bajo y paso alto de 1er orden ---
fc = 1000  # Hz
wc = 2 * np.pi * fc
H0 = 1.0  # ganancia unitaria

f = np.logspace(1, 5, 1000)
w = 2 * np.pi * f

# Funciones de transferencia
H_lp = H0 / (1 + 1j * w / wc)
H_hp = H0 * (1j * w / wc) / (1 + 1j * w / wc)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8))

# Magnitud
ax1.semilogx(f, 20*np.log10(np.abs(H_lp)), color=COLOR_PRINCIPAL, lw=2.5, label='Paso bajo')
ax1.semilogx(f, 20*np.log10(np.abs(H_hp)), color=COLOR_RECTA, lw=2.5, label='Paso alto')
ax1.axvline(fc, color='gray', ls='--', alpha=0.6, label=f'$f_c$ = {fc} Hz')
ax1.axhline(-3, color='gray', ls=':', alpha=0.5)
ax1.annotate('-3 dB', xy=(fc, -3), xytext=(fc*3, -6),
            fontsize=11, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5))
ax1.set_ylabel('Magnitud (dB)')
ax1.set_title('Diagrama de Bode - Filtros de 1er orden', fontweight='bold')
ax1.legend(loc='best')
ax1.set_ylim(-40, 5)
ax1.grid(True, which='both', alpha=0.3)

# Fase
ax2.semilogx(f, np.degrees(np.angle(H_lp)), color=COLOR_PRINCIPAL, lw=2.5, label='Paso bajo')
ax2.semilogx(f, np.degrees(np.angle(H_hp)), color=COLOR_RECTA, lw=2.5, label='Paso alto')
ax2.axvline(fc, color='gray', ls='--', alpha=0.6)
ax2.axhline(-45, color='gray', ls=':', alpha=0.5)
ax2.set_ylabel('Fase (grados)')
ax2.set_xlabel('Frecuencia (Hz)')
ax2.legend(loc='best')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Filtros de segundo orden: funcion de transferencia general

La forma estandar de un filtro paso bajo de segundo orden es:

$$H(s) = \frac{H_0 \cdot \omega_n^2}{s^2 + \frac{\omega_n}{Q} s + \omega_n^2}$$

donde:
- $\omega_n$ = frecuencia natural (rad/s)
- $Q$ = **factor de calidad** (quality factor)
- $\zeta = 1/(2Q)$ = factor de amortiguamiento

**Relacion Q - respuesta:**

| Q | Comportamiento |
|---|---------------|
| $Q < 0.707$ | Sobre-amortiguado (sin pico) |
| $Q = 0.707$ | Butterworth (maximally flat) |
| $Q > 0.707$ | Sub-amortiguado (pico de resonancia) |
| $Q \to \infty$ | Oscilacion |

El **pico de resonancia** ocurre en $\omega_p = \omega_n\sqrt{1 - 1/(2Q^2)}$ con ganancia:

$$|H(j\omega_p)| = \frac{H_0 Q}{\sqrt{1 - 1/(4Q^2)}}$$

Pendiente de caida: **-40 dB/decada** (filtro de 2o orden).

In [ ]:
# --- Efecto del factor Q en la respuesta de un filtro LP de 2o orden ---
fn = 1000  # Hz
wn = 2 * np.pi * fn
f = np.logspace(1, 5, 2000)
w = 2 * np.pi * f

Q_values = [0.5, 0.707, 1.0, 2.0, 5.0]
colors_q = ['#a6bddb', COLOR_BUTTER, '#fdae6b', COLOR_RECTA, COLOR_ACCENT]

fig, ax = plt.subplots(figsize=(11, 6))

for Q, col in zip(Q_values, colors_q):
    H = wn**2 / (-(w**2) + 1j*w*wn/Q + wn**2)
    mag_dB = 20 * np.log10(np.abs(H))
    lbl = f'Q = {Q}' if Q != 0.707 else f'Q = 0.707 (Butterworth)'
    lw = 3.0 if Q == 0.707 else 2.0
    ax.semilogx(f, mag_dB, color=col, lw=lw, label=lbl)

ax.axvline(fn, color='gray', ls='--', alpha=0.5, label=f'$f_n$ = {fn} Hz')
ax.axhline(-3, color='gray', ls=':', alpha=0.4)
ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Magnitud (dB)')
ax.set_title('Efecto del factor de calidad Q en filtro LP de 2o orden', fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(-50, 20)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Aproximacion Butterworth (Maximally Flat)

La respuesta en magnitud al cuadrado de un filtro Butterworth de orden $n$ es:

$$|H(j\omega)|^2 = \frac{1}{1 + \left(\frac{\omega}{\omega_c}\right)^{2n}}$$

**Propiedades:**
- **Maximally flat** en la banda de paso (todas las derivadas nulas en $\omega = 0$)
- Monotona en banda de paso y en banda de rechazo
- A $\omega = \omega_c$: siempre $|H| = -3$ dB independientemente del orden
- Pendiente asintotica: $-20n$ dB/decada
- Polos uniformemente espaciados en un semicirculo de radio $\omega_c$ en el semiplano izquierdo
- Para orden 2: $Q = 1/\sqrt{2} \approx 0.707$

**Ubicacion de polos** (filtro paso bajo normalizado):

$$s_k = \omega_c \cdot e^{j\pi(2k+n-1)/(2n)}, \quad k = 1, 2, \ldots, n$$

(Solo se toman los polos con parte real negativa)

In [ ]:
# --- Butterworth: efecto del orden n ---
f_norm = np.logspace(-1, 1, 2000)  # frecuencia normalizada w/wc

fig, ax = plt.subplots(figsize=(11, 6))

ordenes = [1, 2, 3, 4, 6, 8]
colors_n = ['#a6bddb', '#74a9cf', COLOR_BUTTER, '#045a8d', COLOR_ACCENT, COLOR_RECTA]

for n, col in zip(ordenes, colors_n):
    H2 = 1.0 / (1.0 + f_norm**(2*n))
    mag_dB = 10 * np.log10(H2)
    ax.semilogx(f_norm, mag_dB, color=col, lw=2.5, label=f'n = {n}')

ax.axvline(1.0, color='gray', ls='--', alpha=0.5, label=r'$\omega/\omega_c = 1$')
ax.axhline(-3, color='gray', ls=':', alpha=0.4)
ax.set_xlabel(r'Frecuencia normalizada $\omega/\omega_c$')
ax.set_ylabel('Magnitud (dB)')
ax.set_title('Filtro Butterworth: efecto del orden n', fontweight='bold')
ax.legend(loc='lower left', fontsize=10)
ax.set_ylim(-80, 5)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Aproximacion Chebyshev (Equiripple)

La respuesta Chebyshev Tipo I:

$$|H(j\omega)|^2 = \frac{1}{1 + \varepsilon^2 T_n^2(\omega/\omega_c)}$$

donde $T_n(x)$ es el **polinomio de Chebyshev** de grado $n$ y $\varepsilon$ controla el rizado.

**Propiedades:**
- **Equiripple** en la banda de paso: el rizado oscila entre $1$ y $1/(1+\varepsilon^2)$
- Transicion mas rapida que Butterworth para el mismo orden
- Rizado maximo en banda de paso: $R_p = 10\log_{10}(1+\varepsilon^2)$ dB
- Rizado tipico: 0.5 dB, 1 dB, 3 dB
- Peor respuesta de fase (mayor distorsion de grupo)

**Polinomios de Chebyshev:**
- $T_0(x) = 1$
- $T_1(x) = x$
- $T_2(x) = 2x^2 - 1$
- $T_n(x) = 2x T_{n-1}(x) - T_{n-2}(x)$

**Compromiso:** Mayor pendiente en transicion $\Leftrightarrow$ Mayor rizado en banda de paso.

In [ ]:
# --- Chebyshev Tipo I: efecto del rizado ---
f_norm = np.logspace(-1, 1, 2000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: diferentes rizados con n=4
n = 4
for eps, rp_dB, col, ls in [(0.349, 0.5, COLOR_BUTTER, '-'),
                             (0.509, 1.0, COLOR_CHEBY, '-'),
                             (0.998, 3.0, COLOR_ACCENT, '-')]:
    Tn = np.polynomial.chebyshev.chebval(f_norm, [0]*n + [1])  # T_n(x)
    H2 = 1.0 / (1.0 + eps**2 * Tn**2)
    mag_dB = 10 * np.log10(np.clip(H2, 1e-15, None))
    ax1.semilogx(f_norm, mag_dB, color=col, lw=2.5, ls=ls,
                 label=f'$R_p$ = {rp_dB} dB ($\\varepsilon$ = {eps:.3f})')

ax1.axvline(1.0, color='gray', ls='--', alpha=0.5)
ax1.set_xlabel(r'$\omega/\omega_c$')
ax1.set_ylabel('Magnitud (dB)')
ax1.set_title(f'Chebyshev Tipo I (n={n}): efecto del rizado', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(-60, 5)
ax1.grid(True, which='both', alpha=0.3)

# Derecha: Butterworth vs Chebyshev (n=4, 1dB)
eps_cheby = 0.509
for n_filt in [4]:
    # Butterworth
    H2_b = 1.0 / (1.0 + f_norm**(2*n_filt))
    ax2.semilogx(f_norm, 10*np.log10(H2_b), color=COLOR_BUTTER, lw=2.5,
                 label=f'Butterworth n={n_filt}')
    # Chebyshev
    Tn = np.polynomial.chebyshev.chebval(f_norm, [0]*n_filt + [1])
    H2_c = 1.0 / (1.0 + eps_cheby**2 * Tn**2)
    ax2.semilogx(f_norm, 10*np.log10(np.clip(H2_c, 1e-15, None)), color=COLOR_CHEBY, lw=2.5,
                 label=f'Chebyshev n={n_filt} ($R_p$=1dB)')

ax2.axvline(1.0, color='gray', ls='--', alpha=0.5)
ax2.set_xlabel(r'$\omega/\omega_c$')
ax2.set_ylabel('Magnitud (dB)')
ax2.set_title('Comparacion: Butterworth vs Chebyshev', fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim(-60, 5)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Aproximacion Bessel (Fase lineal)

El filtro Bessel optimiza la **fase lineal** (retardo de grupo constante) en la banda de paso.

$$H(s) = \frac{\theta_n(0)}{\theta_n(s/\omega_0)}$$

donde $\theta_n(s)$ son los **polinomios de Bessel inversos**.

**Propiedades:**
- **Retardo de grupo** practicamente constante en la banda de paso
- Respuesta al escalon sin sobre-pico (overshoot)
- Transicion mas suave (la mas lenta de las tres aproximaciones)
- Ideal para senales con informacion en el dominio temporal

**Comparacion de las tres aproximaciones** (para el mismo orden):

| Propiedad | Butterworth | Chebyshev | Bessel |
|-----------|:-----------:|:---------:|:------:|
| Planitud en banda de paso | Maxima | Rizado | Buena |
| Pendiente de transicion | Media | La mejor | La peor |
| Linealidad de fase | Buena | Mala | La mejor |
| Overshoot en escalon | Moderado | Alto | Minimo |

In [ ]:
# --- Comparacion Butterworth vs Chebyshev vs Bessel usando scipy.signal ---
n = 4
wc = 2 * np.pi * 1000  # 1 kHz

# Diseno de filtros analogicos
b_but, a_but = butter(n, wc, btype='low', analog=True)
b_che, a_che = cheby1(n, 1, wc, btype='low', analog=True)  # 1 dB rizado
b_bes, a_bes = bessel(n, wc, btype='low', analog=True, norm='mag')

# Respuesta en frecuencia
w_eval = np.logspace(1, 5, 2000) * 2 * np.pi
w1, H_but = freqs(b_but, a_but, w_eval)
w2, H_che = freqs(b_che, a_che, w_eval)
w3, H_bes = freqs(b_bes, a_bes, w_eval)

f_hz = w_eval / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Magnitud
for H, col, lbl in [(H_but, COLOR_BUTTER, 'Butterworth'),
                     (H_che, COLOR_CHEBY, 'Chebyshev (1dB)'),
                     (H_bes, COLOR_BESSEL, 'Bessel')]:
    ax1.semilogx(f_hz, 20*np.log10(np.abs(H)), color=col, lw=2.5, label=lbl)

ax1.axvline(1000, color='gray', ls='--', alpha=0.5, label='$f_c$ = 1 kHz')
ax1.axhline(-3, color='gray', ls=':', alpha=0.4)
ax1.set_ylabel('Magnitud (dB)')
ax1.set_title(f'Comparacion de filtros LP de orden n={n}', fontweight='bold', fontsize=14)
ax1.legend(loc='lower left')
ax1.set_ylim(-80, 5)
ax1.grid(True, which='both', alpha=0.3)

# Fase
for H, col, lbl in [(H_but, COLOR_BUTTER, 'Butterworth'),
                     (H_che, COLOR_CHEBY, 'Chebyshev (1dB)'),
                     (H_bes, COLOR_BESSEL, 'Bessel')]:
    ax2.semilogx(f_hz, np.degrees(np.angle(H)), color=col, lw=2.5, label=lbl)

ax2.axvline(1000, color='gray', ls='--', alpha=0.5)
ax2.set_ylabel('Fase (grados)')
ax2.set_xlabel('Frecuencia (Hz)')
ax2.legend(loc='lower left')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Topologia Sallen-Key

La topologia **Sallen-Key** (tambien llamada VCVS - Voltage Controlled Voltage Source) es la topologia mas popular para filtros activos de 2o orden.

### 7.1 Sallen-Key paso bajo

Componentes: R1, R2, C1, C2 + op-amp en configuracion de ganancia unitaria (o con ganancia K).

**Funcion de transferencia (ganancia unitaria, K=1):**

$$H(s) = \frac{1/(R_1 R_2 C_1 C_2)}{s^2 + \left(\frac{1}{R_1 C_1} + \frac{1}{R_2 C_1}\right)s + \frac{1}{R_1 R_2 C_1 C_2}}$$

**Parametros:**
$$\omega_n = \frac{1}{\sqrt{R_1 R_2 C_1 C_2}}$$

$$Q = \frac{\sqrt{R_1 R_2 C_1 C_2}}{R_1 C_1 + R_2 C_1} = \frac{\sqrt{R_1 R_2 C_1 C_2}}{C_1(R_1 + R_2)}$$

### 7.2 Caso simplificado: R1 = R2 = R, C1 = C2 = C

$$\omega_n = \frac{1}{RC}, \quad Q = \frac{1}{2}$$

Este Q = 0.5 es demasiado bajo para Butterworth ($Q_{Butt} = 0.707$). Para lograr Butterworth se necesita ganancia $K = 3 - 1/Q = 1.586$.

In [ ]:
# --- Diagrama del circuito Sallen-Key paso bajo (con matplotlib) ---
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-1, 13)
ax.set_ylim(-2, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Topologia Sallen-Key: Filtro Paso Bajo de 2o Orden', fontweight='bold', fontsize=14)

# Funcion auxiliar para dibujar resistencia
def draw_resistor(ax, x0, y0, x1, y1, label='', color=COLOR_PRINCIPAL):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='-', color=color, lw=2))
    dx, dy = x1-x0, y1-y0
    mid_x, mid_y = (x0+x1)/2, (y0+y1)/2
    rect = mpatches.FancyBboxPatch((mid_x-0.6, mid_y-0.2), 1.2, 0.4,
                                    boxstyle='round,pad=0.05', facecolor='white',
                                    edgecolor=color, lw=2)
    ax.add_patch(rect)
    if label:
        ax.text(mid_x, mid_y+0.55, label, ha='center', va='bottom',
                fontsize=12, fontweight='bold', color=color)

# Funcion auxiliar para dibujar condensador
def draw_capacitor(ax, x, y_top, y_bot, label='', color=COLOR_RECTA):
    gap = 0.15
    ax.plot([x, x], [y_top, y_top - 0.3 + gap], color=color, lw=2)
    ax.plot([x-0.4, x+0.4], [y_top - 0.3 + gap, y_top - 0.3 + gap], color=color, lw=3)
    ax.plot([x-0.4, x+0.4], [y_top - 0.3 - gap, y_top - 0.3 - gap], color=color, lw=3)
    ax.plot([x, x], [y_top - 0.3 - gap, y_bot], color=color, lw=2)
    if label:
        ax.text(x+0.55, (y_top + y_bot)/2, label, ha='left', va='center',
                fontsize=12, fontweight='bold', color=color)

# Nodo de entrada
ax.text(-0.5, 5, r'$V_{in}$', fontsize=14, fontweight='bold', ha='center', va='center')
ax.plot([0, 1], [5, 5], color=COLOR_PRINCIPAL, lw=2)

# R1
draw_resistor(ax, 1, 5, 3.5, 5, r'$R_1$')
ax.plot([3.5, 5], [5, 5], color=COLOR_PRINCIPAL, lw=2)

# R2
draw_resistor(ax, 5, 5, 7.5, 5, r'$R_2$')
ax.plot([7.5, 8], [5, 5], color=COLOR_PRINCIPAL, lw=2)

# C1 (de nodo entre R2 y op-amp a tierra)
draw_capacitor(ax, 5, 5, 2.5, r'$C_1$')
ax.plot([4.5, 5.5], [2.5, 2.5], color='gray', lw=2)  # tierra
ax.plot([4.7, 5.3], [2.3, 2.3], color='gray', lw=1.5)
ax.plot([4.85, 5.15], [2.1, 2.1], color='gray', lw=1)

# Op-amp (triangulo)
tri_x = [8, 10.5, 8, 8]
tri_y = [6.2, 5, 3.8, 6.2]
ax.fill(tri_x, tri_y, color=COLOR_FONDO, edgecolor='black', lw=2.5)
ax.text(7.6, 5.7, '+', fontsize=16, fontweight='bold', ha='center', va='center')
ax.text(7.6, 4.3, '-', fontsize=16, fontweight='bold', ha='center', va='center')

# Conexion V+ del op-amp
ax.plot([8, 8], [5, 5.7], color=COLOR_PRINCIPAL, lw=2)

# Salida op-amp
ax.plot([10.5, 12], [5, 5], color=COLOR_PUNTO, lw=2)
ax.text(12.3, 5, r'$V_{out}$', fontsize=14, fontweight='bold', ha='left', va='center',
        color=COLOR_PUNTO)

# Realimentacion: C2 del nodo entre R1-R2 a salida
ax.plot([3.5, 3.5], [5, 7], color=COLOR_RECTA, lw=2)
ax.plot([3.5, 11], [7, 7], color=COLOR_RECTA, lw=2)
ax.plot([11, 11], [7, 5], color=COLOR_RECTA, lw=2)
ax.plot([11, 10.5], [5, 5], color=COLOR_RECTA, lw=2)
# Simbolo C2 en la linea horizontal
ax.plot([6.8-0.15, 6.8-0.15], [6.6, 7.4], color=COLOR_RECTA, lw=3)
ax.plot([6.8+0.15, 6.8+0.15], [6.6, 7.4], color=COLOR_RECTA, lw=3)
ax.plot([5.5, 6.8-0.15], [7, 7], color=COLOR_RECTA, lw=2)
ax.plot([6.8+0.15, 8.5], [7, 7], color=COLOR_RECTA, lw=2)
ax.text(6.8, 7.6, r'$C_2$', fontsize=12, fontweight='bold', color=COLOR_RECTA, ha='center')

# V- conectado a salida (realimentacion unitaria)
ax.plot([8, 8], [4.3, 3.2], color='gray', lw=2)
ax.plot([8, 11], [3.2, 3.2], color='gray', lw=2)
ax.plot([11, 11], [3.2, 5], color='gray', lw=2)

# Nodos
for nx, ny in [(3.5, 5), (5, 5), (8, 5), (11, 5)]:
    ax.plot(nx, ny, 'o', color='black', ms=5, zorder=5)

# Formulas
ax.text(6, 0.8, r'$\omega_n = \frac{1}{\sqrt{R_1 R_2 C_1 C_2}}$',
        fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=COLOR_FONDO, edgecolor=COLOR_PRINCIPAL, lw=1.5))
ax.text(6, -0.5, r'$Q = \frac{\sqrt{R_1 R_2 C_1 C_2}}{C_1(R_1 + R_2)}$',
        fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=COLOR_FONDO, edgecolor=COLOR_PRINCIPAL, lw=1.5))

plt.tight_layout()
plt.show()

---
## 8. Sensibilidad

La **sensibilidad** mide como varia un parametro del filtro ante variaciones de un componente:

$$S_x^P = \frac{x}{P} \cdot \frac{\partial P}{\partial x} = \frac{\partial \ln P}{\partial \ln x}$$

Interpretacion: si $S_{R_1}^{\omega_n} = -0.5$, un incremento del 1% en $R_1$ produce una disminucion del 0.5% en $\omega_n$.

### Sensibilidades del Sallen-Key paso bajo (K=1):

| Parametro | $S_{R_1}$ | $S_{R_2}$ | $S_{C_1}$ | $S_{C_2}$ |
|-----------|:---------:|:---------:|:---------:|:---------:|
| $\omega_n$ | $-1/2$ | $-1/2$ | $-1/2$ | $-1/2$ |
| $Q$ | Depende de valores | Depende de valores | Depende de valores | Depende de valores |

**Regla practica:** Las sensibilidades del Sallen-Key son tipicamente $\leq 1$ en magnitud, lo cual lo hace una topologia robusta.

Para el caso $R_1 = R_2 = R$, $C_1 = C_2 = C$:
$$S_R^{\omega_n} = -\frac{1}{2}, \quad S_C^{\omega_n} = -\frac{1}{2}$$
$$S_R^Q = 0, \quad S_C^Q = 0$$

(Q no depende de los valores absolutos cuando todos son iguales)

In [ ]:
# --- Analisis de sensibilidad: variacion de wn y Q con tolerancia de componentes ---
np.random.seed(42)
N_montecarlo = 5000
tol = 0.05  # tolerancia 5%

# Valores nominales
R1_nom = 10e3   # 10 kOhm
R2_nom = 10e3
C1_nom = 10e-9  # 10 nF
C2_nom = 10e-9

wn_nom = 1.0 / np.sqrt(R1_nom * R2_nom * C1_nom * C2_nom)
Q_nom = np.sqrt(R1_nom * R2_nom * C1_nom * C2_nom) / (C1_nom * (R1_nom + R2_nom))
fn_nom = wn_nom / (2 * np.pi)

# Monte Carlo
R1_mc = R1_nom * (1 + tol * np.random.uniform(-1, 1, N_montecarlo))
R2_mc = R2_nom * (1 + tol * np.random.uniform(-1, 1, N_montecarlo))
C1_mc = C1_nom * (1 + tol * np.random.uniform(-1, 1, N_montecarlo))
C2_mc = C2_nom * (1 + tol * np.random.uniform(-1, 1, N_montecarlo))

wn_mc = 1.0 / np.sqrt(R1_mc * R2_mc * C1_mc * C2_mc)
Q_mc = np.sqrt(R1_mc * R2_mc * C1_mc * C2_mc) / (C1_mc * (R1_mc + R2_mc))
fn_mc = wn_mc / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Histograma fn
ax1.hist(fn_mc, bins=50, color=COLOR_PRINCIPAL, alpha=0.7, edgecolor='white', density=True)
ax1.axvline(fn_nom, color=COLOR_RECTA, lw=2.5, ls='--', label=f'Nominal: {fn_nom:.0f} Hz')
ax1.set_xlabel('$f_n$ (Hz)')
ax1.set_ylabel('Densidad')
ax1.set_title(f'Distribucion de $f_n$ (tolerancia {tol*100:.0f}%)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histograma Q
ax2.hist(Q_mc, bins=50, color=COLOR_PUNTO, alpha=0.7, edgecolor='white', density=True)
ax2.axvline(Q_nom, color=COLOR_RECTA, lw=2.5, ls='--', label=f'Nominal: Q = {Q_nom:.3f}')
ax2.set_xlabel('Q')
ax2.set_ylabel('Densidad')
ax2.set_title(f'Distribucion de Q (tolerancia {tol*100:.0f}%)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

print(f'fn nominal: {fn_nom:.1f} Hz, sigma: +/- ~{fn_nom*tol/np.sqrt(2):.1f} Hz')
print(f'Q nominal:  {Q_nom:.4f}')

plt.tight_layout()
plt.show()

---
## 9. Comparacion de respuestas: Butterworth vs Chebyshev vs Bessel

Resumen visual de las tres aproximaciones mas utilizadas con las mismas especificaciones.

In [ ]:
# --- Detalle en banda de paso (zoom) ---
n = 4
wc = 2 * np.pi * 1000

b_but, a_but = butter(n, wc, analog=True)
b_che, a_che = cheby1(n, 1, wc, analog=True)
b_bes, a_bes = bessel(n, wc, analog=True, norm='mag')

# Rango estrecho para ver banda de paso
w_pass = np.linspace(0.01, 1.5, 5000) * wc
f_pass = w_pass / (2 * np.pi)

_, H_b = freqs(b_but, a_but, w_pass)
_, H_c = freqs(b_che, a_che, w_pass)
_, H_bs = freqs(b_bes, a_bes, w_pass)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(f_pass, 20*np.log10(np.abs(H_b)), color=COLOR_BUTTER, lw=2.5, label='Butterworth')
ax.plot(f_pass, 20*np.log10(np.abs(H_c)), color=COLOR_CHEBY, lw=2.5, label='Chebyshev (1dB)')
ax.plot(f_pass, 20*np.log10(np.abs(H_bs)), color=COLOR_BESSEL, lw=2.5, label='Bessel')

ax.axhline(-3, color='gray', ls=':', alpha=0.5, label='-3 dB')
ax.axhline(-1, color=COLOR_CHEBY, ls=':', alpha=0.3)
ax.axhline(0, color='gray', ls='-', alpha=0.2)
ax.axvline(1000, color='gray', ls='--', alpha=0.4)

ax.fill_between([0, 1000], -4, 1, alpha=0.05, color=COLOR_PRINCIPAL)
ax.text(500, 0.3, 'Banda de paso', fontsize=11, ha='center', color=COLOR_PRINCIPAL, fontstyle='italic')

ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Magnitud (dB)')
ax.set_title(f'Detalle en banda de paso - Filtros LP orden n={n}', fontweight='bold')
ax.legend(loc='lower left')
ax.set_xlim(0, 1500)
ax.set_ylim(-4, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Respuesta al escalon

La respuesta al escalon revela el comportamiento temporal del filtro, incluyendo:
- **Tiempo de subida** ($t_r$): tiempo para pasar del 10% al 90%
- **Sobre-pico** (overshoot): exceso sobre el valor final
- **Tiempo de establecimiento** ($t_s$): tiempo para entrar en la banda del 2%

In [ ]:
# --- Respuesta al escalon: comparacion de los tres filtros ---
from scipy.signal import step

n = 4
wc = 2 * np.pi * 1000

# Sistemas analogicos
sys_but = signal.lti(*butter(n, wc, analog=True))
sys_che = signal.lti(*cheby1(n, 1, wc, analog=True))
sys_bes = signal.lti(*bessel(n, wc, analog=True, norm='mag'))

t_max = 3e-3  # 3 ms
t_eval = np.linspace(0, t_max, 2000)

fig, ax = plt.subplots(figsize=(11, 6))

for sys, col, lbl in [(sys_but, COLOR_BUTTER, 'Butterworth'),
                       (sys_che, COLOR_CHEBY, 'Chebyshev (1dB)'),
                       (sys_bes, COLOR_BESSEL, 'Bessel')]:
    t_out, y_out = step(sys, T=t_eval)
    ax.plot(t_out * 1e3, y_out, color=col, lw=2.5, label=lbl)

ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
ax.axhline(1.02, color='gray', ls=':', alpha=0.3)
ax.axhline(0.98, color='gray', ls=':', alpha=0.3)
ax.fill_between([0, t_max*1e3], 0.98, 1.02, alpha=0.08, color='gray')
ax.text(2.5, 1.05, 'Banda 2%', fontsize=10, ha='center', color='gray', fontstyle='italic')

ax.set_xlabel('Tiempo (ms)')
ax.set_ylabel('Amplitud')
ax.set_title(f'Respuesta al escalon - Filtros LP de orden n={n}', fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, t_max*1e3)
ax.set_ylim(-0.1, 1.4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. Seleccion del orden del filtro

Dado un conjunto de especificaciones:
- Frecuencia de corte $f_c$ (-3 dB)
- Atenuacion minima $A_{min}$ (dB) a frecuencia $f_s$ (stopband)

El orden minimo requerido para Butterworth es:

$$n \geq \frac{\log\left(10^{A_{min}/10} - 1\right)}{2 \log(f_s / f_c)}$$

Para Chebyshev Tipo I:

$$n \geq \frac{\cosh^{-1}\left(\sqrt{\frac{10^{A_{min}/10} - 1}{\varepsilon^2}}\right)}{\cosh^{-1}(f_s / f_c)}$$

In [ ]:
# --- Grafico de seleccion de orden: atenuacion vs frecuencia para diferentes ordenes ---
f_ratio = np.linspace(1, 10, 500)  # fs/fc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Butterworth
for n in [1, 2, 3, 4, 6, 8]:
    atten_dB = 10 * np.log10(1 + f_ratio**(2*n))
    ax1.plot(f_ratio, atten_dB, lw=2, label=f'n = {n}')

ax1.axhline(40, color=COLOR_RECTA, ls='--', alpha=0.5, label='Spec: 40 dB')
ax1.axhline(60, color=COLOR_ACCENT, ls='--', alpha=0.5, label='Spec: 60 dB')
ax1.set_xlabel(r'$f_s / f_c$')
ax1.set_ylabel('Atenuacion (dB)')
ax1.set_title('Butterworth: atenuacion vs $f_s/f_c$', fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3)

# Calculo automatico del orden minimo
Amin_values = [20, 30, 40, 50, 60]
fs_fc_values = np.linspace(1.1, 5, 200)

for Amin in Amin_values:
    n_min = np.ceil(np.log(10**(Amin/10) - 1) / (2 * np.log(fs_fc_values)))
    ax2.plot(fs_fc_values, n_min, lw=2, label=f'$A_{{min}}$ = {Amin} dB')

ax2.set_xlabel(r'$f_s / f_c$')
ax2.set_ylabel('Orden minimo n (Butterworth)')
ax2.set_title('Orden minimo requerido', fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_ylim(0, 12)
ax2.set_yticks(range(1, 13))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 12. Ubicacion de polos en el plano s

Los polos determinan la estabilidad y comportamiento del filtro. Para un filtro estable, todos los polos deben estar en el **semiplano izquierdo** (Re(s) < 0).

### Patrones de polos:
- **Butterworth:** polos sobre un semicirculo de radio $\omega_c$, uniformemente espaciados
- **Chebyshev:** polos sobre una elipse (parte real comprimida)
- **Bessel:** polos distribuidos de forma no uniforme, mas a la izquierda

In [ ]:
# --- Ubicacion de polos en el plano s para los tres tipos ---
n = 4
wc = 2 * np.pi * 1000

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

filtros = [
    ('Butterworth', butter(n, wc, analog=True), COLOR_BUTTER),
    ('Chebyshev (1dB)', cheby1(n, 1, wc, analog=True), COLOR_CHEBY),
    ('Bessel', bessel(n, wc, analog=True, norm='mag'), COLOR_BESSEL),
]

for ax, (nombre, (b, a), col) in zip(axes, filtros):
    z, p, k = tf2zpk(b, a)
    
    # Semicirculo de referencia
    theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(wc * np.cos(theta), wc * np.sin(theta), 'k:', alpha=0.2, lw=1)
    
    # Ejes
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    
    # Polos
    ax.plot(np.real(p), np.imag(p), 'x', color=col, ms=12, mew=3, label='Polos')
    
    # Ceros (si existen)
    if len(z) > 0:
        ax.plot(np.real(z), np.imag(z), 'o', color=col, ms=10, mfc='none', mew=2, label='Ceros')
    
    ax.set_xlabel(r'Re(s) (rad/s)')
    ax.set_ylabel(r'Im(s) (rad/s)')
    ax.set_title(nombre, fontweight='bold', color=col)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Formato de ejes
    lim = wc * 1.5
    ax.set_xlim(-lim, lim * 0.3)
    ax.set_ylim(-lim, lim)

plt.suptitle(f'Mapa de polos - Filtros LP de orden n={n}', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## 13. Ejercicios resueltos

### Ejercicio 1: Diseno de filtro Butterworth

**Enunciado:** Disenar un filtro paso bajo Butterworth con:
- $f_c = 1$ kHz (frecuencia de corte -3 dB)
- Atenuacion minima: 40 dB a 4 kHz

Determinar: (a) orden minimo, (b) funcion de transferencia, (c) respuesta en frecuencia.

In [ ]:
# === EJERCICIO 1: Diseno filtro Butterworth ===
fc = 1000  # Hz
fs = 4000  # Hz
Amin = 40  # dB

# (a) Orden minimo
n_exact = np.log(10**(Amin/10) - 1) / (2 * np.log(fs/fc))
n = int(np.ceil(n_exact))
print(f'=== Ejercicio 1: Filtro Butterworth LP ===')
print(f'Especificaciones: fc = {fc} Hz, fs = {fs} Hz, Amin = {Amin} dB')
print(f'Orden minimo (exacto): {n_exact:.2f}')
print(f'Orden seleccionado: n = {n}')

# (b) Funcion de transferencia
wc = 2 * np.pi * fc
b, a = butter(n, wc, analog=True)
print(f'\nCoeficientes del numerador:   {b}')
print(f'Coeficientes del denominador: longitud = {len(a)}')

# Verificacion
w_check = 2 * np.pi * fs
_, H_check = freqs(b, a, [w_check])
atten_check = -20 * np.log10(np.abs(H_check[0]))
print(f'\nVerificacion a {fs} Hz: atenuacion = {atten_check:.1f} dB (req: {Amin} dB)')
print(f'  -> {"CUMPLE" if atten_check >= Amin else "NO CUMPLE"}')

# (c) Respuesta en frecuencia
w_plot = np.logspace(1, 5, 2000) * 2 * np.pi
_, H = freqs(b, a, w_plot)
f_plot = w_plot / (2 * np.pi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7))

ax1.semilogx(f_plot, 20*np.log10(np.abs(H)), color=COLOR_BUTTER, lw=2.5)
ax1.axvline(fc, color='gray', ls='--', alpha=0.5, label=f'$f_c$ = {fc} Hz')
ax1.axvline(fs, color=COLOR_RECTA, ls='--', alpha=0.5, label=f'$f_s$ = {fs} Hz')
ax1.axhline(-3, color='gray', ls=':', alpha=0.4)
ax1.axhline(-Amin, color=COLOR_RECTA, ls=':', alpha=0.4, label=f'-{Amin} dB')
ax1.fill_between([fs, 1e5], -100, 0, alpha=0.05, color=COLOR_RECTA)
ax1.set_ylabel('Magnitud (dB)')
ax1.set_title(f'Ejercicio 1: Butterworth LP, n={n}', fontweight='bold')
ax1.legend(loc='lower left')
ax1.set_ylim(-80, 5)
ax1.grid(True, which='both', alpha=0.3)

ax2.semilogx(f_plot, np.degrees(np.angle(H)), color=COLOR_BUTTER, lw=2.5)
ax2.axvline(fc, color='gray', ls='--', alpha=0.5)
ax2.set_ylabel('Fase (grados)')
ax2.set_xlabel('Frecuencia (Hz)')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

### Ejercicio 2: Sallen-Key Butterworth

**Enunciado:** Disenar un filtro Sallen-Key paso bajo de 2o orden tipo Butterworth con $f_c = 5$ kHz.

Elegir $C_1 = C_2 = C = 10$ nF. Determinar $R_1$, $R_2$ y verificar Q.

In [ ]:
# === EJERCICIO 2: Sallen-Key Butterworth ===
fc_target = 5000  # Hz
wn_target = 2 * np.pi * fc_target
Q_target = 1.0 / np.sqrt(2)  # 0.707 para Butterworth

C = 10e-9  # 10 nF (eleccion)

print('=== Ejercicio 2: Sallen-Key LP Butterworth ===')
print(f'Especificaciones: fc = {fc_target} Hz, Q = {Q_target:.4f} (Butterworth)')
print(f'C1 = C2 = C = {C*1e9:.0f} nF')

# Con R1=R2=R, C1=C2=C y ganancia unitaria:
#   wn = 1/(RC) -> R = 1/(wn*C)
#   Q = 1/2 (insuficiente para Butterworth)
#   Necesitamos ganancia K = 3 - 1/Q = 1.586

print('\n--- Metodo 1: R1=R2=R, C1=C2=C con ganancia K ---')
K = 3.0 - 1.0/Q_target
R = 1.0 / (wn_target * C)
print(f'R = 1/(wn*C) = {R:.1f} Ohm = {R/1e3:.2f} kOhm')
print(f'Ganancia necesaria: K = 3 - 1/Q = {K:.4f}')
print(f'Resistencias de ganancia: K = 1 + RB/RA = {K:.3f}')
print(f'  Si RA = 10 kOhm -> RB = {(K-1)*10:.3f} kOhm = {(K-1)*10e3:.0f} Ohm')

print('\n--- Metodo 2: R1=R2=R, C1 != C2 (ganancia unitaria) ---')
# Con R1=R2=R y C1!=C2: Q = sqrt(C2/C1)/2
# Para Q = 0.707: C2/C1 = (2Q)^2 = 2
ratio_C = (2*Q_target)**2
C2_new = 10e-9
C1_new = C2_new / ratio_C
R_new = 1.0 / (wn_target * np.sqrt(C1_new * C2_new))
Q_check = np.sqrt(R_new**2 * C1_new * C2_new) / (C1_new * 2 * R_new)
print(f'C2/C1 = {ratio_C:.1f}')
print(f'C1 = {C1_new*1e9:.1f} nF, C2 = {C2_new*1e9:.1f} nF')
print(f'R = R1 = R2 = {R_new:.1f} Ohm = {R_new/1e3:.3f} kOhm')
print(f'Q verificado = {Q_check:.4f} (objetivo: {Q_target:.4f})')

# Verificacion fn
wn_real = 1.0 / (R_new * np.sqrt(C1_new * C2_new))
print(f'fn verificada = {wn_real/(2*np.pi):.0f} Hz (objetivo: {fc_target} Hz)')

### Ejercicio 3: Chebyshev vs Butterworth - Comparacion de orden

**Enunciado:** Se requiere un filtro LP con:
- $f_c = 2$ kHz
- Atenuacion $\geq$ 50 dB a $f_s = 6$ kHz

Comparar el orden necesario para Butterworth y Chebyshev (1 dB rizado).

In [ ]:
# === EJERCICIO 3: Comparacion de orden Butterworth vs Chebyshev ===
fc = 2000  # Hz
fs = 6000  # Hz
Amin = 50  # dB
Rp = 1.0   # dB rizado Chebyshev

print('=== Ejercicio 3: Comparacion de orden ===')
print(f'Especificaciones: fc = {fc} Hz, fs = {fs} Hz, Amin = {Amin} dB')

# Orden Butterworth
n_but = int(np.ceil(np.log(10**(Amin/10) - 1) / (2 * np.log(fs/fc))))
print(f'\nButterworth: n = {n_but}')

# Orden Chebyshev
eps = np.sqrt(10**(Rp/10) - 1)
n_che = int(np.ceil(np.arccosh(np.sqrt((10**(Amin/10)-1)/eps**2)) / np.arccosh(fs/fc)))
print(f'Chebyshev (Rp={Rp}dB): n = {n_che}')
print(f'\nAhorro de orden: {n_but - n_che} etapa(s) menos con Chebyshev')

# Verificacion con scipy
wc = 2 * np.pi * fc

b_b, a_b = butter(n_but, wc, analog=True)
b_c, a_c = cheby1(n_che, Rp, wc, analog=True)

w_plot = np.logspace(1, 5, 3000) * 2 * np.pi
_, H_b = freqs(b_b, a_b, w_plot)
_, H_c = freqs(b_c, a_c, w_plot)
f_plot = w_plot / (2 * np.pi)

# Verificar atenuacion
_, H_b_check = freqs(b_b, a_b, [2*np.pi*fs])
_, H_c_check = freqs(b_c, a_c, [2*np.pi*fs])
print(f'\nVerificacion a fs={fs} Hz:')
print(f'  Butterworth (n={n_but}): {-20*np.log10(np.abs(H_b_check[0])):.1f} dB')
print(f'  Chebyshev   (n={n_che}): {-20*np.log10(np.abs(H_c_check[0])):.1f} dB')

fig, ax = plt.subplots(figsize=(11, 6))
ax.semilogx(f_plot, 20*np.log10(np.abs(H_b)), color=COLOR_BUTTER, lw=2.5,
            label=f'Butterworth n={n_but}')
ax.semilogx(f_plot, 20*np.log10(np.abs(H_c)), color=COLOR_CHEBY, lw=2.5,
            label=f'Chebyshev n={n_che} (1dB)')

ax.axvline(fc, color='gray', ls='--', alpha=0.5, label=f'$f_c$ = {fc} Hz')
ax.axvline(fs, color='orange', ls='--', alpha=0.5, label=f'$f_s$ = {fs} Hz')
ax.axhline(-Amin, color=COLOR_RECTA, ls=':', alpha=0.5, label=f'-{Amin} dB')
ax.fill_between([fs, 1e5], -100, -Amin, alpha=0.06, color=COLOR_PUNTO)

ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Magnitud (dB)')
ax.set_title('Ejercicio 3: Butterworth vs Chebyshev - mismo requisito', fontweight='bold')
ax.legend(loc='lower left')
ax.set_ylim(-80, 5)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicio 4: Filtro paso banda con cascada LP + HP

**Enunciado:** Disenar un filtro paso banda con:
- Frecuencia central: $f_0 = 1$ kHz
- Ancho de banda: $BW = 200$ Hz
- Q = $f_0 / BW = 5$

Implementar como cascada de LP + HP de 2o orden.

In [ ]:
# === EJERCICIO 4: Filtro paso banda LP+HP en cascada ===
f0 = 1000  # Hz
BW = 200   # Hz
Q_bp = f0 / BW
fL = f0 - BW/2  # 900 Hz
fH = f0 + BW/2  # 1100 Hz

print('=== Ejercicio 4: Filtro paso banda ===')
print(f'f0 = {f0} Hz, BW = {BW} Hz, Q = {Q_bp}')
print(f'fL (HP) = {fL} Hz, fH (LP) = {fH} Hz')

wL = 2 * np.pi * fL
wH = 2 * np.pi * fH

# LP de 2o orden en fH
b_lp, a_lp = butter(2, wH, btype='low', analog=True)
# HP de 2o orden en fL
b_hp, a_hp = butter(2, wL, btype='high', analog=True)

# Cascada: multiplicar funciones de transferencia
b_bp = np.polymul(b_lp, b_hp)
a_bp = np.polymul(a_lp, a_hp)

w_plot = np.logspace(1, 5, 5000) * 2 * np.pi
f_plot = w_plot / (2 * np.pi)

_, H_lp = freqs(b_lp, a_lp, w_plot)
_, H_hp = freqs(b_hp, a_hp, w_plot)
_, H_bp = freqs(b_bp, a_bp, w_plot)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8))

# Magnitud
ax1.semilogx(f_plot, 20*np.log10(np.abs(H_lp)), color=COLOR_BUTTER, lw=2, ls='--',
             alpha=0.6, label=f'LP (fc={fH} Hz)')
ax1.semilogx(f_plot, 20*np.log10(np.abs(H_hp)), color=COLOR_CHEBY, lw=2, ls='--',
             alpha=0.6, label=f'HP (fc={fL} Hz)')
ax1.semilogx(f_plot, 20*np.log10(np.abs(H_bp)), color=COLOR_ACCENT, lw=3,
             label='Paso banda (cascada)')

ax1.axvline(f0, color='gray', ls=':', alpha=0.5, label=f'$f_0$ = {f0} Hz')
ax1.axvline(fL, color=COLOR_CHEBY, ls=':', alpha=0.3)
ax1.axvline(fH, color=COLOR_BUTTER, ls=':', alpha=0.3)
ax1.axhline(-3, color='gray', ls=':', alpha=0.3)
ax1.set_ylabel('Magnitud (dB)')
ax1.set_title('Ejercicio 4: Filtro paso banda (LP + HP en cascada)', fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.set_ylim(-60, 5)
ax1.grid(True, which='both', alpha=0.3)

# Fase
ax2.semilogx(f_plot, np.degrees(np.angle(H_bp)), color=COLOR_ACCENT, lw=2.5)
ax2.axvline(f0, color='gray', ls=':', alpha=0.5)
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel('Fase (grados)')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 14. Catalogo de tipos de ejercicio

Referencia rapida de los tipos de problemas mas comunes en el tema de filtros activos.

### Tipo 1: Calculo del orden minimo (Butterworth)

**Datos:** $f_c$, $f_s$, $A_{min}$

**Formula:** $n \geq \dfrac{\log(10^{A_{min}/10} - 1)}{2\log(f_s/f_c)}$, redondear hacia arriba.

---

### Tipo 2: Calculo del orden minimo (Chebyshev)

**Datos:** $f_c$, $f_s$, $A_{min}$, $R_p$ (rizado)

**Formula:** $\varepsilon = \sqrt{10^{R_p/10}-1}$, luego $n \geq \dfrac{\cosh^{-1}\sqrt{(10^{A_{min}/10}-1)/\varepsilon^2}}{\cosh^{-1}(f_s/f_c)}$

---

### Tipo 3: Diseno Sallen-Key LP

**Datos:** $f_c$ y tipo de aproximacion (Butterworth/Chebyshev)

**Procedimiento:**
1. Fijar C (o R) con valor comercial
2. Calcular $R = 1/(\omega_c C)$ si $R_1 = R_2$
3. Ajustar ganancia K o ratio $C_2/C_1$ para obtener Q deseado
4. Para Butterworth: $K = 3 - 1/Q = 1.586$

---

### Tipo 4: Analisis de sensibilidad

**Datos:** Circuito Sallen-Key con valores de componentes

**Procedimiento:**
1. Calcular $S_x^{\omega_n}$ y $S_x^Q$ para cada componente
2. Estimar variacion: $\Delta P/P \approx S_x^P \cdot \Delta x / x$

---

### Tipo 5: Paso banda / Rechazo de banda

**Datos:** $f_0$, $BW$ (o $Q$), atenuacion requerida

**Procedimiento:**
1. Calcular frecuencias de corte: $f_L = f_0 - BW/2$, $f_H = f_0 + BW/2$
2. Disenar LP en $f_H$ y HP en $f_L$
3. Cascadear ambos filtros
4. Para rechazo de banda: LP($f_L$) + HP($f_H$) en paralelo y sumar

---
## 15. Resumen y tabla de formulas clave

In [ ]:
# --- Tabla resumen visual ---
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)

ax.text(7, 9.6, 'RESUMEN: Filtros Activos', fontsize=18, fontweight='bold',
        ha='center', va='center', color=COLOR_PRINCIPAL)

# Bloques de formulas
bloques = [
    (1.5, 8.2, 'LP 1er orden',
     r'$H(s) = \frac{H_0}{1 + s/\omega_c}$' + '\n' + r'Roll-off: -20 dB/dec',
     COLOR_PRINCIPAL),
    (5.5, 8.2, 'LP 2o orden (general)',
     r'$H(s) = \frac{H_0 \omega_n^2}{s^2 + \frac{\omega_n}{Q}s + \omega_n^2}$' + '\n' + r'Roll-off: -40 dB/dec',
     COLOR_PRINCIPAL),
    (10, 8.2, 'Orden n',
     r'Roll-off: $-20n$ dB/dec' + '\n' + r'$|H(\omega_c)|$ = -3 dB',
     COLOR_PRINCIPAL),
    (1.5, 6.0, 'Butterworth',
     r'$|H|^2 = \frac{1}{1+(\omega/\omega_c)^{2n}}$' + '\n' + 'Maximally flat',
     COLOR_BUTTER),
    (5.5, 6.0, 'Chebyshev Tipo I',
     r'$|H|^2 = \frac{1}{1+\varepsilon^2 T_n^2(\omega/\omega_c)}$' + '\n' + 'Equiripple en paso',
     COLOR_CHEBY),
    (10, 6.0, 'Bessel',
     'Fase lineal\nRetardo grupo cte\nSin overshoot',
     COLOR_BESSEL),
    (3.5, 3.8, 'Sallen-Key LP',
     r'$\omega_n = \frac{1}{\sqrt{R_1 R_2 C_1 C_2}}$' + '\n' +
     r'$Q = \frac{\sqrt{R_1 R_2 C_1 C_2}}{C_1(R_1+R_2)}$',
     COLOR_ACCENT),
    (8.5, 3.8, 'Sensibilidad',
     r'$S_x^P = \frac{x}{P}\frac{\partial P}{\partial x}$' + '\n' +
     r'$\Delta P/P \approx S_x^P \cdot \Delta x/x$',
     COLOR_ACCENT),
    (3.5, 1.5, 'Orden Butterworth',
     r'$n \geq \frac{\log(10^{A/10}-1)}{2\log(f_s/f_c)}$',
     COLOR_BUTTER),
    (8.5, 1.5, 'Butterworth Q (2o orden)',
     r'$Q = 1/\sqrt{2} \approx 0.707$' + '\n' + r'SK: $K = 3-\sqrt{2} \approx 1.586$',
     COLOR_BUTTER),
]

for x, y, titulo, formula, color in bloques:
    rect = FancyBboxPatch((x-1.6, y-0.9), 3.2, 1.7,
                           boxstyle='round,pad=0.15',
                           facecolor='white', edgecolor=color, lw=2)
    ax.add_patch(rect)
    ax.text(x, y+0.45, titulo, fontsize=10, fontweight='bold',
            ha='center', va='center', color=color)
    ax.text(x, y-0.25, formula, fontsize=9, ha='center', va='center',
            color='black', linespacing=1.6)

plt.tight_layout()
plt.show()